离线构造教学语言视图。核心代码、已有输出与配置沿已接受读者源；固定申报事实/图属于历史教学输入。下列新计算不构成市场策略复现。

[本语言逐本运行模式与准备条件](../NOTEBOOK_GUIDE.md)。本阶段只制作语言层，便携环境与执行在后续隔离冷启动核验。

In [ ]:
from pathlib import Path
import os, sys
from uuid import uuid4
ROOT = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (p / "pyproject.toml").is_file() and (p / "src/quant_course").is_dir())
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "code"))
os.chdir(ROOT)
os.environ["MPLBACKEND"] = "Agg"
RUN_DIR = ROOT / "runs" / ("ch34_market_microstructure_" + uuid4().hex)
RUN_DIR.mkdir(parents=True, exist_ok=False)
print("Default offline teaching; current empirical French execution SKIPPED (permissions unresolved).")
print("Run directory:", RUN_DIR)


# 第34章：单场所静态订单簿扫单

**本Notebook中的订单簿和数值都是合成教学构造。** 它演示一个静态快照中的可见深度计算，不是交易所撮合器，也没有使用真实行情、外部下载、随机数或历史执行证据。

价格以USD/股计，数量以股计；价格使用Decimal，数量使用整数。

In [ ]:
from decimal import Decimal
from pathlib import Path
from quant_course.microstructure import Level, OrderBook, sweep_displayed_book

book = OrderBook(
    bids=(Level(Decimal("100.00"), 300), Level(Decimal("99.98"), 400)),
    asks=(Level(Decimal("100.02"), 200), Level(Decimal("100.04"), 300)),
)
mid = book.midpoint()
spread = book.quoted_spread()
buy = sweep_displayed_book(book, "buy", 350)
sell = sweep_displayed_book(book, "sell", 350)

print({"best_bid": str(book.bids[0].price), "best_ask": str(book.asks[0].price),
       "mid": str(mid), "spread": str(spread)})
print("buy fills:", [(str(x.price), x.size) for x in buy.fills],
      "notional:", buy.notional, "VWAP:", buy.vwap)
print("hypothetical sell fills:", [(str(x.price), x.size) for x in sell.fills],
      "proceeds:", sell.notional, "VWAP:", sell.vwap)

In [ ]:
assert mid == Decimal("100.01")
assert spread == Decimal("0.02")
assert [(x.price, x.size) for x in buy.fills] == [
    (Decimal("100.02"), 200), (Decimal("100.04"), 150)
]
assert buy.notional == Decimal("35010.00")
assert buy.vwap == Decimal("35010") / 350
assert [(x.price, x.size) for x in sell.fills] == [
    (Decimal("100.00"), 300), (Decimal("99.98"), 50)
]
assert sell.notional == Decimal("34999.00")
arrival_mid_mark_pnl = Decimal(buy.filled) * mid - buy.notional
round_trip_pnl = sell.notional - buy.notional
assert arrival_mid_mark_pnl == Decimal("-6.50")
assert round_trip_pnl == Decimal("-11.00")
print({"arrival_mid_mark_pnl_usd": arrival_mid_mark_pnl,
       "hypothetical_round_trip_pnl_usd": round_trip_pnl})

## 如何解释结果

中间价USD 100.01是参考标记，不是有卖方承诺成交的订单。350股买单跨过两个ask价位；代码只对快照中显示的股数计算，并假设它们在读取时都可成交。往返PnL是假设随后立即在同一冻结簿卖回，不能当成真实历史路径或可重复市场状态。

簿内走动和报价点差进入成交价格；手续费、延迟、队列、隐藏数量、撤单、补单、路由和市场响应均不在模型中。没有对超过显示深度的剩余数量编造价格。

In [ ]:
thin_book = OrderBook(
    bids=(),
    asks=(Level(Decimal("100.02"), 100),),
)
partial = sweep_displayed_book(thin_book, "buy", 350)
empty = sweep_displayed_book(OrderBook(bids=(), asks=()), "buy", 20)
assert (partial.filled, partial.remaining, partial.vwap) == (
    100, 250, Decimal("100.02")
)
assert (empty.filled, empty.remaining, empty.vwap) == (0, 20, None)
print({"partial_fill": partial.filled, "uncovered": partial.remaining,
       "no_fill_vwap": empty.vwap})

## 时间契约与OHLCV边界

下列UTC时间是字段关系的**构造示例**，不是观察数据。event_time记市场事件；available_at记研究流程最早可用时刻；ingested_at记本地接收；decision_time之后才可发送订单。撮合回放还需要场所接收、确认、逐事件状态和成交时间。日线OHLCV没有这些信息，不能证明限价触价就成交或还原队列。

In [ ]:
from datetime import datetime, timezone

clock_example = {
    "event_time": datetime.fromisoformat("2026-10-08T14:30:00+00:00"),
    "available_at": datetime.fromisoformat("2026-10-08T14:30:00.050000+00:00"),
    "ingested_at": datetime.fromisoformat("2026-10-08T14:30:00.080000+00:00"),
    "decision_time": datetime.fromisoformat("2026-10-08T14:30:00.100000+00:00"),
    "order_send_time": datetime.fromisoformat("2026-10-08T14:30:00.110000+00:00"),
    "order_receive_time": datetime.fromisoformat("2026-10-08T14:30:00.120000+00:00"),
    "fill_time": datetime.fromisoformat("2026-10-08T14:30:00.125000+00:00"),
}
assert all(value.tzinfo == timezone.utc for value in clock_example.values())
assert clock_example["available_at"] <= clock_example["decision_time"]
assert clock_example["decision_time"] < clock_example["order_send_time"]
assert clock_example["order_send_time"] <= clock_example["order_receive_time"]
assert clock_example["order_receive_time"] <= clock_example["fill_time"]
print({key: value.isoformat() for key, value in clock_example.items()})

In [ ]:
import runpy
from IPython.display import Image, display

root = Path.cwd()
script = runpy.run_path(str(root / "code/ch34_market_microstructure.py"))
figure_path = RUN_DIR / "figures/ch34_order_book_sweep.png"
script["plot_case"](book, figure_path)
display(Image(filename=str(figure_path)))

## 复核问题

1. 为什么arrival-mid mark PnL与假设往返PnL不同？
2. 若最佳ask仅有100股，订单请求350股，哪些数量可计算，哪些还需要额外场所/订单规则？
3. 哪些event-level字段能帮助重建订单状态？哪些字段不能由日线OHLCV补出？

图中左侧横轴是每档显示股数、纵轴是USD/股；右侧以USD显示静态构造的往返分解。它不包含真实市场数据。